# MCP client and server

**Model Context Protocol (MCP)** connects an AI application to external **tools** (actions) and **resources** (read-only data) through a small server process.

```
╔══════════════════════════════════════════════════════════════════════════════╗
║           MODEL CONTEXT PROTOCOL (MCP) — what it is, at a glance             ║
╚══════════════════════════════════════════════════════════════════════════════╝

   ┌─────────────────────┐         JSON-RPC over transport          ┌─────────────────────┐
   │     MCP HOST        │ ◄────────────────────────────────────► │     MCP SERVER      │
   │  Cursor, notebook,  │    stdio (this lab) · HTTP · SSE       │  mcp_demo_server.py │
   │  LangGraph agent    │                                        │  separate process   │
   └──────────┬──────────┘                                        └──────────┬──────────┘
              │ uses                                                         │ exposes
              ▼                                                              ▼
   ┌─────────────────────┐                                        ┌─────────────────────┐
   │   MCP CLIENT SDK    │                                        │   CAPABILITIES      │
   │  ClientSession      │                                        │                     │
   │  • initialize()     │                                        │  TOOLS  = do things │──► search, escalate, …
   │  • list_tools()     │                                        │  RESOURCES = read   │──► JSON, runbook text
   │  • call_tool()      │                                        │                     │
   │  • read_resource()  │                                        └──────────┬──────────┘
   └─────────────────────┘                                                   │
                                                                              ▼
                                                                   ┌─────────────────────┐
                                                                   │  BACKEND / DATA     │
                                                                   │  DB, APIs, files    │
                                                                   │  (demo: in-memory)  │
                                                                   └─────────────────────┘

   KEY IDEA: the host discovers capabilities at runtime (list_tools / list_resources)
   instead of hard-coding every integration inside the agent.
```

This lab uses an **L1 ticketing** demo with:

| Capability | Examples in this server |
|------------|-------------------------|
| **Tools** | Search incidents, fetch runbooks, escalate to L2 |
| **Resources** | JSON list of open incidents, per-category runbook text |

**Why MCP?** Separation of concerns, reuse across clients, runtime discovery via `list_tools()` / `list_resources()`, and a clear safety boundary around what the agent can access.

The client cells below discover capabilities, read a resource, then run a short **multi-step workflow** similar to how an agent would handle a database outage ticket.

## Server

The server (`mcp_demo_server.py`) exposes four tools and two resource URIs:

| Tool | Purpose |
|------|--------|
| `search_open_incidents` | Keyword search across open incidents |
| `lookup_ticket_status` | Incidents for a ticket category |
| `get_runbook_steps` | Numbered troubleshooting steps |
| `escalate_incident` | Hand off to L2 and record escalation id |

| Resource URI | Purpose |
|--------------|--------|
| `ticketing://incidents/open` | JSON snapshot of open incidents |
| `ticketing://runbook/{category}` | Runbook text for database, network, or auth |

```
╔══════════════════════════════════════════════════════════════════════════════╗
║     OUR DEMO — mcp_demo_server.py (FastMCP, transport=stdio)                 ║
╚══════════════════════════════════════════════════════════════════════════════╝

  JUPYTER NOTEBOOK (MCP client)              SUBPROCESS: mcp_demo_server.py
  ┌──────────────────────────┐  stdin/stdout  ┌────────────────────────────────┐
  │ stdio_client()           │ ◄────────────► │ FastMCP("l1-ticketing")        │
  │ ClientSession            │  JSON-RPC      │                                │
  │ • list_tools()           │                │  IN-MEMORY DATA (demo only)    │
  │ • call_tool(name, args)  │                │  ┌────────────┐ ┌───────────┐  │
  │ • read_resource(uri)     │                │  │ _INCIDENTS │ │ _RUNBOOKS │  │
  └──────────────────────────┘                │  │ INC-1042   │ │ database  │  │
                                              │  │ INC-2087   │ │ network   │  │
                                              │  │ INC-3101   │ │ auth      │  │
                                              │  └─────┬──────┘ └─────┬─────┘  │
                                              │        │              │        │
                                              │  ┌─────▼──────────────▼─────┐  │
                                              │  │ TOOLS (actions)          │  │
                                              │  │ search_open_incidents    │  │
                                              │  │ lookup_ticket_status     │  │
                                              │  │ get_runbook_steps        │  │
                                              │  │ escalate_incident ────────┼──┼──► _ESCALATIONS[]
                                              │  └──────────────────────────┘  │
                                              │  ┌──────────────────────────┐  │
                                              │  │ RESOURCES (read-only)    │  │
                                              │  │ ticketing://incidents/open│  │
                                              │  │ ticketing://runbook/{cat}│  │
                                              │  └──────────────────────────┘  │
                                              └────────────────────────────────┘

  TOOLS change state (e.g. escalate marks INC-1042 as ESCALATED).
  RESOURCES return snapshots without side effects — good for context injection.
```

Run the next cell to write the server file, then the client cells start it as a subprocess over **stdio**.

In [7]:
%%writefile mcp_demo_server.py
"""L1 ticketing MCP demo server — tools + resources for labs/day-4/2-mcp-basics.ipynb."""
from __future__ import annotations

import json
import uuid
from datetime import datetime, timezone

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("l1-ticketing")

_INCIDENTS: list[dict] = [
    {
        "id": "INC-1042",
        "category": "database",
        "status": "OPEN",
        "severity": "high",
        "summary": "DB connection pool exhaustion under load",
        "opened_at": "2026-06-28T09:15:00Z",
    },
    {
        "id": "INC-2087",
        "category": "network",
        "status": "MONITORING",
        "severity": "medium",
        "summary": "Intermittent latency on east gateway",
        "opened_at": "2026-06-29T14:02:00Z",
    },
    {
        "id": "INC-3101",
        "category": "auth",
        "status": "OPEN",
        "severity": "low",
        "summary": "Spike in failed login attempts after deploy",
        "opened_at": "2026-06-30T07:40:00Z",
    },
]

_RUNBOOKS: dict[str, list[str]] = {
    "database": [
        "Check Hikari/pool active vs idle connections",
        "Search logs for 'Connection is not available' or pool timeout",
        "Compare pool max size with concurrent request peak",
        "If sustained, scale pool or reduce connection hold time",
    ],
    "network": [
        "Ping east gateway and upstream health endpoints",
        "Review firewall and CDN cache hit ratio",
        "Check for recent config or certificate changes",
    ],
    "auth": [
        "Verify identity provider status page",
        "Review lockout thresholds and recent deploy",
        "Clear stale sessions if token refresh is failing",
    ],
}

_ESCALATIONS: list[dict] = []


def _now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


@mcp.tool()
def search_open_incidents(keyword: str, limit: int = 5) -> str:
    """Search OPEN/MONITORING incidents where id, category, or summary contains keyword."""
    needle = keyword.lower()
    hits = [
        i
        for i in _INCIDENTS
        if i["status"] in ("OPEN", "MONITORING")
        and (
            needle in i["id"].lower()
            or needle in i["category"].lower()
            or needle in i["summary"].lower()
        )
    ][:limit]
    if not hits:
        return f"No open incidents matching '{keyword}'."
    lines = [
        f"{h['id']} [{h['category']}] {h['status']} ({h['severity']}): {h['summary']}"
        for h in hits
    ]
    return "Open incidents:\n" + "\n".join(lines)


@mcp.tool()
def lookup_ticket_status(subject: str, category: str) -> str:
    """Return open incidents for a ticket category (database, network, auth)."""
    cat = category.lower()
    matches = [
        i for i in _INCIDENTS if i["category"] == cat and i["status"] in ("OPEN", "MONITORING")
    ]
    if not matches:
        return f"Lookup '{subject}' [{category}]: No related open incident found."
    lines = [
        f"{m['id']} {m['status']} ({m['severity']}): {m['summary']}" for m in matches
    ]
    return f"Lookup '{subject}' [{category}]:\n" + "\n".join(lines)


@mcp.tool()
def get_runbook_steps(category: str) -> str:
    """Return numbered troubleshooting steps for a category."""
    steps = _RUNBOOKS.get(category.lower())
    if not steps:
        return f"No runbook for category '{category}'. Try: database, network, auth."
    numbered = "\n".join(f"{n}. {step}" for n, step in enumerate(steps, start=1))
    return f"Runbook ({category}):\n{numbered}"


@mcp.tool()
def escalate_incident(incident_id: str, reason: str, team: str = "L2-oncall") -> str:
    """Escalate an incident to another on-call team. Returns a new escalation reference."""
    incident = next((i for i in _INCIDENTS if i["id"] == incident_id), None)
    if not incident:
        return f"Cannot escalate: unknown incident '{incident_id}'."
    esc_id = f"ESC-{uuid.uuid4().hex[:8].upper()}"
    record = {
        "escalation_id": esc_id,
        "incident_id": incident_id,
        "team": team,
        "reason": reason,
        "created_at": _now_iso(),
    }
    _ESCALATIONS.append(record)
    incident["status"] = "ESCALATED"
    return (
        f"Escalated {incident_id} -> {team} as {esc_id}. Reason: {reason}"
    )


@mcp.resource("ticketing://incidents/open")
def open_incidents_resource() -> str:
    """JSON snapshot of all OPEN and MONITORING incidents."""
    open_items = [i for i in _INCIDENTS if i["status"] in ("OPEN", "MONITORING")]
    return json.dumps(open_items, indent=2)


@mcp.resource("ticketing://runbook/{category}")
def runbook_resource(category: str) -> str:
    """Plain-text runbook for a category (database, network, auth)."""
    steps = _RUNBOOKS.get(category.lower(), [])
    if not steps:
        return f"No runbook for '{category}'."
    return "\n".join(f"{n}. {s}" for n, s in enumerate(steps, start=1))


if __name__ == "__main__":
    mcp.run(transport="stdio")


Overwriting mcp_demo_server.py


## Client helpers

The MCP Python SDK is **async** (tool calls wait on subprocess I/O). We wrap repeated patterns in small helpers and use `errlog=sys.__stderr__` so Windows/Jupyter keeps server logs off the notebook stdout stream.

In [8]:
import os
import sys

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

errlog = sys.__stderr__ if hasattr(sys.__stderr__, "fileno") else open(os.devnull, "w")
SERVER_PARAMS = StdioServerParameters(command=sys.executable, args=["mcp_demo_server.py"])


def tool_text(result) -> str:
    """Join text blocks returned by call_tool."""
    return "\n".join(getattr(b, "text", str(b)) for b in result.content)


async def call_tool(session, name: str, **arguments) -> str:
    result = await session.call_tool(name, arguments=arguments)
    return tool_text(result)


async def read_resource_text(session, uri: str) -> str:
    body = await session.read_resource(uri)
    return body.contents[0].text

print("Helpers ready.")


Helpers ready.


## 1. Discovery — list tools and resources

An agent (or IDE) typically calls `list_tools()` and `list_resources()` before choosing what to invoke. Template resources use `list_resource_templates()`.

In [9]:
async with stdio_client(SERVER_PARAMS, errlog=errlog) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        tools = await session.list_tools()
        print("Tools:", [t.name for t in tools.tools])

        resources = await session.list_resources()
        print("Resources:", [str(r.uri) for r in resources.resources])

        templates = await session.list_resource_templates()
        print("Resource templates:", [t.uriTemplate for t in templates.resourceTemplates])

        snapshot = await read_resource_text(session, "ticketing://incidents/open")
        print("\nOpen incidents (first 200 chars):")
        print(snapshot[:200], "...")


Tools: ['search_open_incidents', 'lookup_ticket_status', 'get_runbook_steps', 'escalate_incident']
Resources: ['ticketing://incidents/open']
Resource templates: ['ticketing://runbook/{category}']

Open incidents (first 200 chars):
[
  {
    "id": "INC-1042",
    "category": "database",
    "status": "OPEN",
    "severity": "high",
    "summary": "DB connection pool exhaustion under load",
    "opened_at": "2026-06-28T09:15:00Z" ...


## 2. Multi-step workflow — handle a database ticket

Simulate what an L1 agent might do for: *App returns 500s; logs show pool timeout.*

```
  CLIENT WORKFLOW (database ticket demo)
  ─────────────────────────────────────

  Customer ticket: "App 500 errors after traffic spike"  [category: database]
              │
              ▼
  ┌─────────────────────────┐
  │ 1. search_open_incidents│  keyword="pool"  ──► finds INC-1042
  └────────────┬────────────┘
               ▼
  ┌─────────────────────────┐
  │ 2. lookup_ticket_status │  category=database ──► confirms related incident
  └────────────┬────────────┘
               ▼
  ┌─────────────────────────┐     ┌─────────────────────────┐
  │ 3a. read_resource       │     │ 3b. get_runbook_steps   │
  │ ticketing://runbook/... │     │ (same steps via tool)   │
  └────────────┬────────────┘     └────────────┬────────────┘
               └──────────────┬─────────────────┘
                              ▼
  ┌─────────────────────────┐
  │ 4. escalate_incident      │  INC-1042 → L2-database + ESC-xxxx id
  └─────────────────────────┘
```

1. **Search** incidents for `pool`
2. **Lookup** by category `database`
3. **Read runbook** resource and **call** `get_runbook_steps` tool
4. **Escalate** high-severity incident to L2 if still open

In [10]:
TICKET_SUBJECT = "App 500 errors after traffic spike"
TICKET_CATEGORY = "database"

async with stdio_client(SERVER_PARAMS, errlog=errlog) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        print("=== Step 1: search_open_incidents ===")
        print(await call_tool(session, "search_open_incidents", keyword="pool", limit=3))

        print("\n=== Step 2: lookup_ticket_status ===")
        print(await call_tool(session, "lookup_ticket_status", subject=TICKET_SUBJECT, category=TICKET_CATEGORY))

        print("\n=== Step 3a: read_resource runbook ===")
        print(await read_resource_text(session, f"ticketing://runbook/{TICKET_CATEGORY}"))

        print("\n=== Step 3b: get_runbook_steps tool ===")
        print(await call_tool(session, "get_runbook_steps", category=TICKET_CATEGORY))

        print("\n=== Step 4: escalate_incident ===")
        print(await call_tool(
            session,
            "escalate_incident",
            incident_id="INC-1042",
            reason="Sustained 500s; pool exhausted per runbook step 1",
            team="L2-database",
        ))


=== Step 1: search_open_incidents ===
Open incidents:
INC-1042 [database] OPEN (high): DB connection pool exhaustion under load

=== Step 2: lookup_ticket_status ===
Lookup 'App 500 errors after traffic spike' [database]:
INC-1042 OPEN (high): DB connection pool exhaustion under load

=== Step 3a: read_resource runbook ===
1. Check Hikari/pool active vs idle connections
2. Search logs for 'Connection is not available' or pool timeout
3. Compare pool max size with concurrent request peak
4. If sustained, scale pool or reduce connection hold time

=== Step 3b: get_runbook_steps tool ===
Runbook (database):
1. Check Hikari/pool active vs idle connections
2. Search logs for 'Connection is not available' or pool timeout
3. Compare pool max size with concurrent request peak
4. If sustained, scale pool or reduce connection hold time

=== Step 4: escalate_incident ===
Escalated INC-1042 -> L2-database as ESC-AE17C47C. Reason: Sustained 500s; pool exhausted per runbook step 1


## 3. Auth ticket — lookup and runbook only

A second scenario shows a lower-severity path without escalation.

In [11]:
async with stdio_client(SERVER_PARAMS, errlog=errlog) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        print(await call_tool(session, "lookup_ticket_status", subject="Login failures", category="auth"))
        print()
        print(await call_tool(session, "get_runbook_steps", category="auth"))


Lookup 'Login failures' [auth]:
INC-3101 OPEN (low): Spike in failed login attempts after deploy

Runbook (auth):
1. Verify identity provider status page
2. Review lockout thresholds and recent deploy
3. Clear stale sessions if token refresh is failing


### What you should see

- **Discovery**: four tool names; resource `ticketing://incidents/open`; template `ticketing://runbook/{category}`.
- **Workflow**: INC-1042 found for database/pool; runbook steps printed; escalation returns an ESC-... id.
- **Auth scenario**: INC-3101 and auth runbook steps without escalation.
